# Import

In [1]:
!pip -q install transformers accelerate datasets sentencepiece

# Model

In [2]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# 더 가볍게 시작하려면:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("Loaded:", MODEL_NAME)
print("Device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


# Util Func

In [ ]:
import re

def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def clean_answer(text: str) -> str:
    text = text.strip()
    # 첫 줄만 사용
    text = text.split("\n")[0].strip()
    # 코드블록 같은 이상 출력 잘라내기
    text = text.split("```")[0].strip()
    return text

def generate_text(prompt: str, max_new_tokens: int = 128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # temperature/top_p/top_k 안 씀
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

## QA Func

### With Context

In [ ]:
def ask_qa_with_context(context: str, question: str, max_new_tokens: int = 48):
    prompt = f"""다음 문서를 읽고 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[문서]
{context}

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)



### Without Context

In [5]:
def ask_qa_without_context(question: str, max_new_tokens: int = 48):
    prompt = f"""다음 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)

## Predict Search Func

In [ ]:
def predict_need_search(question: str, context: str = "", max_new_tokens: int = 4):
    prompt = f"""당신의 임무는 외부 검색 필요 여부만 판단하는 것입니다.

규칙:
1. 최신 정보, 현재 시점 정보, 자주 바뀌는 사실이면 SEARCH
2. 일반 상식, 오래 변하지 않는 사실이면 NO_SEARCH
3. 반드시 SEARCH 또는 NO_SEARCH 중 하나만 출력
4. 설명 금지

[질문]
{question}

[문맥]
{context if context else "없음"}

[판단]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)

    if "[판단]" in text:
        result = text.split("[판단]")[-1].strip()
    else:
        result = text.strip()

    result = result.split()[0].upper()

    if result == "SEARCH":
        return "SEARCH"
    elif result == "NO_SEARCH":
        return "NO_SEARCH"
    return result

# DataSet

In [7]:
from datasets import load_dataset

dataset = load_dataset("klue", "mrc", split="validation[:100]")
print(dataset[0].keys())

dict_keys(['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'])


In [8]:

ds = load_dataset("klue", "mrc")

print("=== Dataset Summary ===")
print(ds)

print("\n=== Split Sizes ===")
for split_name, split_data in ds.items():
    print(f"{split_name}: {len(split_data)}")

print("\n=== Column Names ===")
for split_name, split_data in ds.items():
    print(f"{split_name}: {split_data.column_names}")

print("\n=== First Sample (train) ===")
print(ds["train"][0])

=== Dataset Summary ===
DatasetDict({
    train: Dataset({
        features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
        num_rows: 17554
    })
    validation: Dataset({
        features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
        num_rows: 5841
    })
})

=== Split Sizes ===
train: 17554
validation: 5841

=== Column Names ===
train: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers']
validation: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers']

=== First Sample (train) ===
{'title': '제주도 장마 시작 … 중부는 이달 말부터', 'context': '올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도 늦은 이달 말께 장마가 시작될 전망이다.17일 기상청에 따르면 제주도 남쪽 먼바다에 있는 장마전선의 영향으로 이날 제주도 산간 및 내륙지역에 호우주의보가 내려지면서 곳곳에 100㎜에 육박하는 많은 비가 내렸다. 제주의 장마는 평년보다 2~3일, 지난해보다는 

## rough match Func

In [ ]:
def rough_match(pred: str, gold: str) -> bool:
    pred_n = normalize_text(pred)
    gold_n = normalize_text(gold)
    return (gold_n in pred_n) or (pred_n in gold_n)

## PipeLine

In [ ]:
def qa_with_search_pipeline(question, context):
    decision = predict_need_search(question, context)

    if decision == "SEARCH":
        pred = ask_qa_with_context(context, question)
    else:
        pred = ask_qa_without_context(question)

    return decision, pred

### PipeLine Test

In [11]:
pipeline_correct = 0
pipeline_results = []

for ex in dataset:
    question = ex["question"]
    context = ex["context"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    decision, pred = qa_with_search_pipeline(question, context)

    ok = rough_match(pred, gold)
    pipeline_correct += int(ok)

    pipeline_results.append({
        "question": question,
        "decision": decision,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

pipeline_acc = pipeline_correct / len(dataset)
print(f"Pipeline accuracy: {pipeline_correct}/{len(dataset)} = {pipeline_acc:.3f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Pipeline accuracy: 62/100 = 0.620


### PipeLine Errors

In [12]:
pipeline_errors = [r for r in pipeline_results if not r["correct"]]
print(f"Pipeline wrong cases: {len(pipeline_errors)}")

for i, case in enumerate(pipeline_errors[:10]):
    print(f"\n[Pipeline Error {i+1}]")
    print("Q:", case["question"])
    print("Decision:", case["decision"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Pipeline wrong cases: 38

[Pipeline Error 1]
Q: 2014년 일하고 싶은 50대 회사 중에서 5위로 선정된 기업은?
Decision: SEARCH
Gold: 페이스북
Pred: 링크트인

[Pipeline Error 2]
Q: 포브스의 2014년 일하고 싶은 50대 회사 조사에서 5위를 한 기업은?
Decision: SEARCH
Gold: 페이스북
Pred: 링크트인

[Pipeline Error 3]
Q: 가공하는데 몇 초밖에 걸리지 않는 물질은?
Decision: SEARCH
Gold: 실리콘유
Pred: 셀룰로오스의 히드록시기와 반응하여 천 위를 덮어서 방수성을 준다. (몇 초만)

[Pipeline Error 4]
Q: 실리콘을 실리콘유로 만들기 위해 거치는 과정은?
Decision: SEARCH
Gold: 실리콘
Pred: 실란을 원료로 사용하여 디메틸폴리실록산으로 만든다

[Pipeline Error 5]
Q: 성서를 영어로 번역한 인물에게 영향을 받은 사람은?
Decision: SEARCH
Gold: 얀 후스
Pred: 존 위클리프

[Pipeline Error 6]
Q: 법무법인 한결이 세워진 지 10년차에 합병된 업체의 이름은?
Decision: SEARCH
Gold: 법무법인 내일
Pred: 한살림

[Pipeline Error 7]
Q: 한국예술영재교육원 원장이 현재 학생을 가르치는 곳은?
Decision: SEARCH
Gold: 한국예술종합학교 음악원
Pred: 한예종 부설 한국예술영재교육원

[Pipeline Error 8]
Q: 허벅지까지 내려오는 미들 기장의 다운 자켓은 이름은?
Decision: NO_SEARCH
Gold: ‘로사다운자켓’
Pred: Mid-Calf Length Down Jacket

[Pipeline Error 9]
Q: 데카메론에서 마을의 모든 부자들에게 돈을 달라고 하는 인물은?
Decision: SEARCH
Gold: 걸인
Pred: 없음

[Pipeline Error 10

In [13]:
under_search = []
qa_failure = []
over_search = []

for r in pipeline_results:
    if not r["correct"]:
        if r["decision"] == "NO_SEARCH":
            under_search.append(r)
        elif r["decision"] == "SEARCH":
            qa_failure.append(r)

print("UNDER_SEARCH:", len(under_search))
print("QA_FAILURE:", len(qa_failure))

UNDER_SEARCH: 5
QA_FAILURE: 33


## 오류 라벨링

In [14]:
from google.colab import drive
import pandas as pd
import os

# 1. 드라이브 연결
drive.mount('/content/drive')

# 2. pipeline 오답 추출
pipeline_errors = []
for idx, r in enumerate(pipeline_results):
    if not r["correct"]:
        pipeline_errors.append({
            "dataset_idx": int(idx),
            **r
        })

print("Total pipeline errors:", len(pipeline_errors))

# 3. 분석할 개수 설정
N = min(20, len(pipeline_errors))
analysis_samples = pipeline_errors[:N]

# 4. 최종 라벨링용 테이블 생성 (context 전체 포함)
analysis_table = pd.DataFrame([
    {
        "row_id": i,
        "dataset_idx": int(case["dataset_idx"]),
        "question": case["question"],
        "decision": case["decision"],
        "gold": case["gold"],
        "pred": case["pred"],
        "all_gold_answers": " | ".join(dataset[int(case["dataset_idx"])]["answers"]["text"]),
        "context_full": dataset[int(case["dataset_idx"])]["context"],   # ✅ 전체 context
        "retrieval_quality": "",   # YES / PARTIAL / NO
        "utilization_type": "",    # IGNORED / PARTIAL_USE / CONTRADICTED / HALLUCINATED / OTHER
        "notes": ""
    }
    for i, case in enumerate(analysis_samples)
])

# 5. 저장 경로
local_path = "./pilot_failure_analysis_full_context.csv"
drive_path = "/content/drive/MyDrive/retrieval-utilization/pilot/pilot_02_failure_analysis_full_context.csv"

# 6. 저장
analysis_table.to_csv(local_path, index=False, encoding="utf-8-sig")
analysis_table.to_csv(drive_path, index=False, encoding="utf-8-sig")

# 7. 확인
print("\n✅ 저장 완료")
print("현재 폴더:", os.getcwd())
print("로컬 저장:", os.path.abspath(local_path))
print("드라이브 저장:", drive_path)
print("\n현재 폴더 파일 목록:")
print(os.listdir("."))

# 8. 미리보기
display(analysis_table.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total pipeline errors: 38

✅ 저장 완료
현재 폴더: /content
로컬 저장: /content/pilot_failure_analysis_full_context.csv
드라이브 저장: /content/drive/MyDrive/retrieval-utilization/pilot/pilot_02_failure_analysis_full_context.csv

현재 폴더 파일 목록:
['.config', 'pilot_failure_analysis_full_context.csv', 'pilot_failure_analysis.csv', 'drive', 'sample_data']


,row_id,dataset_idx,question,decision,gold,pred,all_gold_answers,context_full,retrieval_quality,utilization_type,notes
0,0,3,2014년 일하고 싶은 50대 회사 중에서 5위로 선정된 기업은?,SEARCH,페이스북,링크트인,페이스북,가장 일하고 싶은 회사로 미국 컨설팅 업체인 베인&컴퍼니가 선정됐다. 트위터와 링크...,,,
1,1,4,포브스의 2014년 일하고 싶은 50대 회사 조사에서 5위를 한 기업은?,SEARCH,페이스북,링크트인,페이스북,가장 일하고 싶은 회사로 미국 컨설팅 업체인 베인&컴퍼니가 선정됐다. 트위터와 링크...,,,
2,2,10,가공하는데 몇 초밖에 걸리지 않는 물질은?,SEARCH,실리콘유,셀룰로오스의 히드록시기와 반응하여 천 위를 덮어서 방수성을 준다. (몇 초만),실리콘유,규소 수지(硅素樹脂)는 실리콘 물질의 일종이다.\n\n이산화규소는 1개의 규소 원자...,,,


### 라밸링 기준
#### retrieval_quality
	•	YES: context 안에 정답 단서가 충분히 있음
	•	PARTIAL: 단서는 있는데 불완전하거나 간접적
	•	NO: context 안에 정답 단서가 거의 없음

#### utilization_type
	•	IGNORED: 답이 있었는데 모델이 안 씀
	•	PARTIAL_USE: 일부만 참고해서 틀림
	•	CONTRADICTED: 문서와 반대로 답함 / 다른 entity 선택
	•	HALLUCINATED: 문서와 무관한 답 생성
	•	OTHER: 애매한 경우

In [15]:
analysis_table.loc[0, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context states LinkedIn=3rd and Facebook=5th; model chose 3rd"
]

analysis_table.loc[1, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "same pattern as row 0"
]

analysis_table.loc[2, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "correct clue exists ('실리콘유') but model outputs descriptive phrase instead of entity"
]

analysis_table.loc[3, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "model uses context but fails to extract exact target answer"
]

analysis_table.loc[4, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context states Jan Hus was influenced by Wycliffe; model outputs influencer"
]

analysis_table.loc[5, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly mentions 법무법인 내일; model outputs unrelated merged name"
]

analysis_table.loc[6, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "context contains both current school and gifted center role; model confuses institution"
]

analysis_table.loc[7, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "product name appears explicitly, but model outputs generic English description"
]

analysis_table.loc[8, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "HALLUCINATED", "context explicitly contains '걸인' but model answers '없음'"
]

analysis_table.loc[9, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context distinguishes Cyprus (decision point) from Corfu (later arrival); model chooses later location"
]

analysis_table.loc[10, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly states '독일의 한 마을'; model answers Italy"
]

analysis_table.loc[11, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context explicitly says '보석을 줄테니'; model outputs another phrase"
]

analysis_table.loc[12, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "correct entity is in context, but model outputs corrupted variant of the name"
]

analysis_table.loc[13, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "magic stone is explicit in context; model outputs long descriptive sentence instead"
]

analysis_table.loc[14, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "context explicitly mentions carbon dioxide; model returns functional explanation instead of entity"
]

analysis_table.loc[15, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "IGNORED", "first sentence contains '부서지기 쉬운 뼈'; model ignores explicit evidence"
]

analysis_table.loc[16, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "context explicitly defines 현실색; model outputs descriptive paraphrase instead of exact term"
]

analysis_table.loc[17, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context states restoration was by Alexander III; model answers Nicholas II"
]

analysis_table.loc[18, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "PARTIAL_USE", "model extracts correct city but adds noisy extra text"
]

analysis_table.loc[19, ["retrieval_quality", "utilization_type", "notes"]] = [
    "YES", "CONTRADICTED", "context links '숲속의 정경' to first CD ('오이제비우스'); model answers CD2"
]

analysis_table.head(20)

,row_id,dataset_idx,question,decision,gold,pred,all_gold_answers,context_full,retrieval_quality,utilization_type,notes
0,0,3,2014년 일하고 싶은 50대 회사 중에서 5위로 선정된 기업은?,SEARCH,페이스북,링크트인,페이스북,가장 일하고 싶은 회사로 미국 컨설팅 업체인 베인&컴퍼니가 선정됐다. 트위터와 링크...,YES,CONTRADICTED,context states LinkedIn=3rd and Facebook=5th; ...
1,1,4,포브스의 2014년 일하고 싶은 50대 회사 조사에서 5위를 한 기업은?,SEARCH,페이스북,링크트인,페이스북,가장 일하고 싶은 회사로 미국 컨설팅 업체인 베인&컴퍼니가 선정됐다. 트위터와 링크...,YES,CONTRADICTED,same pattern as row 0
2,2,10,가공하는데 몇 초밖에 걸리지 않는 물질은?,SEARCH,실리콘유,셀룰로오스의 히드록시기와 반응하여 천 위를 덮어서 방수성을 준다. (몇 초만),실리콘유,규소 수지(硅素樹脂)는 실리콘 물질의 일종이다.\n\n이산화규소는 1개의 규소 원자...,YES,PARTIAL_USE,correct clue exists ('실리콘유') but model outputs...
3,3,11,실리콘을 실리콘유로 만들기 위해 거치는 과정은?,SEARCH,실리콘,실란을 원료로 사용하여 디메틸폴리실록산으로 만든다,실리콘,규소 수지(硅素樹脂)는 실리콘 물질의 일종이다.\n\n이산화규소는 1개의 규소 원자...,YES,PARTIAL_USE,model uses context but fails to extract exact ...
4,4,18,성서를 영어로 번역한 인물에게 영향을 받은 사람은?,SEARCH,얀 후스,존 위클리프,얀 후스,에라스무스의 초상\n 르네상스 인문주의 문서를 참고하십시오.\n초기 르네상스 시기 ...,YES,CONTRADICTED,context states Jan Hus was influenced by Wycli...
5,5,21,법무법인 한결이 세워진 지 10년차에 합병된 업체의 이름은?,SEARCH,법무법인 내일,한살림,법무법인 내일 | 내일,“유능한 변호사들을 팀 단위로 영입해 규모의 경제와 전문성을 꾀하겠다.”안식 법무법...,YES,CONTRADICTED,context explicitly mentions 법무법인 내일; model out...
6,6,22,한국예술영재교육원 원장이 현재 학생을 가르치는 곳은?,SEARCH,한국예술종합학교 음악원,한예종 부설 한국예술영재교육원,한국예술종합학교 음악원 | 한국예술종합학교 | 한예종,“제자들이 (바이올린 연주를) 잘하면 제가 잘하는 것보다 100배는 행복해요. 젊었...,YES,PARTIAL_USE,context contains both current school and gifte...
7,7,23,허벅지까지 내려오는 미들 기장의 다운 자켓은 이름은?,NO_SEARCH,‘로사다운자켓’,Mid-Calf Length Down Jacket,‘로사다운자켓’ | 로사다운자켓,언제 어디서나 아이들의 즐거운 활동을 응원하는 블랙야크키즈가 화사한 색감과 사랑스러...,YES,PARTIAL_USE,"product name appears explicitly, but model out..."
8,8,24,데카메론에서 마을의 모든 부자들에게 돈을 달라고 하는 인물은?,SEARCH,걸인,없음,걸인,"나탄은 명망 높은 갑부 노인으로, 한 젊은 부자는 그 노인의 명성에 매우 큰 질투를...",YES,HALLUCINATED,context explicitly contains '걸인' but model ans...
9,9,25,란돌포는 어느 섬에서 해적질을 결심하였는가?,SEARCH,키프로스 섬,코르퓨 섬,키프로스 섬,란돌포는 부자로 무역을 하기 위해 큰 배에 전재산을 털어 상품을 사서 키프로스 섬으...,YES,CONTRADICTED,context distinguishes Cyprus (decision point) ...


In [16]:
print("=== Retrieval Quality Counts ===")
print(analysis_table["retrieval_quality"].value_counts())

print("\n=== Utilization Type Counts ===")
print(analysis_table["utilization_type"].value_counts())

=== Retrieval Quality Counts ===
retrieval_quality
YES    20
Name: count, dtype: int64

=== Utilization Type Counts ===
utilization_type
CONTRADICTED    9
PARTIAL_USE     9
HALLUCINATED    1
IGNORED         1
Name: count, dtype: int64
